# Denoising Autoencoder (DAE) — Multi-Anatomy MRI Anomaly Detection

This notebook trains **three DAE models** for artifact detection:

| # | Anatomy | Dataset | Description |
|---|---------|---------|-------------|
| 1 | **Knee** | FastMRI (preprocessed) | Single anatomy, single contrast |
| 2 | **Brain** | IXI (balanced T1 + T2) | Single anatomy, balanced contrast |
| 3 | **Combined** | Both datasets merged | Multi-anatomy generalist model |

## Architecture (Vincent et al., ICML 2008 / JMLR 2010)
- **Symmetric CNN autoencoder** — no skip connections (forces bottleneck learning)
- **Encoder**: 4 double-conv blocks (1→64→128→256→512) + AdaptiveAvgPool → 512-d bottleneck
- **Decoder**: 4 upsample+conv blocks (512→256→128→64→1) + Sigmoid
- **Noise model**: Gaussian noise σ∈[0.05, 0.25] + random rectangular masking + salt-and-pepper
- **Loss**: MSE (clean target vs denoised output)
- **Balanced brain sampling** — same deterministic subset as MAE/DINO/SimCLR

## Anomaly Scoring (dual)
| Method | Description |
|--------|-------------|
| **Reconstruction** | Avg MSE over 10 noise trials (parallels MAE multi-mask scoring) |
| **kNN (bottleneck)** | Cosine kNN on 512-d encoder features (same as SimCLR/MAE/DINO) |

## Consistency with Other Methods
| Setting | DAE | SimCLR | MAE | DINO |
|---------|-----|--------|-----|------|
| Image size | 192×192 | 192×192 | 192×192 | 192×192 |
| Effective batch | 256 | 256 | 256 | 256 |
| Epochs | 20 | 20 | 20 | 20 |
| Brain sampling | Balanced (same) | Balanced (same) | Balanced (same) | Balanced (same) |
| Evaluation | kNN + recon + collapse + t-SNE | kNN + collapse + t-SNE | kNN + recon + collapse + t-SNE | kNN + collapse + t-SNE |

In [ ]:
import os
import math
import time
import gc
import numpy as np
import random
import matplotlib.pyplot as plt
import scipy.ndimage

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from tqdm import tqdm
from sklearn.neighbors import NearestNeighbors
from sklearn.manifold import TSNE

# ── Reproducibility ──
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
if device == "cuda":
    print(torch.cuda.get_device_name(0))
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Hyperparameters

- **CNN encoder** — 4 blocks, 512-d bottleneck (no skip connections)
- **Noise model** — Gaussian σ∈[0.05, 0.25], rectangular masking, salt-and-pepper
- **Reconstruction trials** — 10 (same as MAE multi-mask scoring)
- **20 epochs** — consistent with SimCLR, MAE, DINO

In [ ]:
# ══════════════════════════════════════════════
# Hyperparameters (shared across all 3 models)
# ══════════════════════════════════════════════
BATCH_SIZE     = 64
GRAD_ACCUM     = 4            # effective batch = 64 × 4 = 256
EPOCHS         = 20           # same as SimCLR, MAE, DINO
BASE_LR        = 1e-3         # AdamW with cosine decay
WARMUP_EPOCHS  = 2
WEIGHT_DECAY   = 1e-4
IMG_SIZE       = 192
BOTTLENECK_DIM = 512          # encoder feature dimension

# DAE-specific
NOISE_STD_LO   = 0.05        # Gaussian noise σ range
NOISE_STD_HI   = 0.25
MASK_PROB       = 0.5         # probability of rectangular masking
MASK_MAX_RATIO  = 0.20        # max fraction of image per mask rectangle
SP_PROB         = 0.3         # probability of salt-and-pepper noise
SP_AMOUNT       = 0.02        # fraction of pixels affected
RECON_TRIALS    = 10          # noise trials for reconstruction scoring (same as MAE)

# Evaluation
K_NEIGHBORS    = 5
NUM_WORKERS    = 4
MAX_GRAD_NORM  = 1.0

print(f"Effective batch size: {BATCH_SIZE * GRAD_ACCUM}")
print(f"Epochs per model: {EPOCHS}")
print(f"Noise: Gaussian σ∈[{NOISE_STD_LO}, {NOISE_STD_HI}] + mask(p={MASK_PROB}) + S&P(p={SP_PROB})")
print(f"Recon trials: {RECON_TRIALS}")

## Data Pipeline

### Balanced Brain Sampling
Same deterministic `get_balanced_paths()` as MAE, DINO, and SimCLR — identical brain subset.

### DAE Noise Corruption
Three noise types (applied to create the noisy input; the clean image is the target):
1. **Gaussian noise** (always) — σ ~ U[0.05, 0.25]
2. **Random rectangular masking** (p=0.5) — 1–3 rectangles, up to 20% of image area each
3. **Salt-and-pepper noise** (p=0.3) — random pixels set to 0 or 1

In [ ]:
# ══════════════════════════════════════════════
# Balanced Sampling + Noise Model + Datasets
# ══════════════════════════════════════════════

def get_balanced_paths(root_dir, target_total=None):
    """Deterministic balanced subsampling — IDENTICAL to MAE/DINO/SimCLR notebooks."""
    groups = {}
    for dirpath, _, filenames in os.walk(root_dir):
        npy_files = sorted(f for f in filenames if f.endswith('.npy'))
        if npy_files:
            folder = os.path.basename(dirpath)
            groups[folder] = [os.path.join(dirpath, f) for f in npy_files]

    if not groups:
        return [], []

    if target_total is None:
        paths, labels = [], []
        for folder in sorted(groups):
            paths.extend(groups[folder])
            labels.extend([folder] * len(groups[folder]))
        return paths, labels

    n_groups = len(groups)
    target_per_group = target_total // n_groups

    paths, labels = [], []
    for folder in sorted(groups):
        group_paths = groups[folder]
        if len(group_paths) <= target_per_group:
            selected = group_paths
        else:
            idx = np.round(np.linspace(0, len(group_paths) - 1, target_per_group)).astype(int)
            selected = [group_paths[i] for i in idx]
        paths.extend(selected)
        labels.extend([folder] * len(selected))
        print(f"    {folder}: {len(selected)}/{len(group_paths)} selected")

    return paths, labels


# ── Noise corruption functions ──

def add_gaussian_noise(img, std_lo=NOISE_STD_LO, std_hi=NOISE_STD_HI):
    sigma = random.uniform(std_lo, std_hi)
    return img + sigma * np.random.randn(*img.shape).astype(np.float32)


def add_rect_mask(img, max_ratio=MASK_MAX_RATIO, num_masks=None):
    h, w = img.shape
    img = img.copy()
    n = num_masks or random.randint(1, 3)
    for _ in range(n):
        mh = int(h * random.uniform(0.05, max_ratio))
        mw = int(w * random.uniform(0.05, max_ratio))
        y = random.randint(0, h - mh)
        x = random.randint(0, w - mw)
        img[y:y+mh, x:x+mw] = 0.0
    return img


def add_salt_pepper(img, amount=SP_AMOUNT):
    img = img.copy()
    n_pixels = img.size
    n_salt = int(n_pixels * amount / 2)
    n_pepper = int(n_pixels * amount / 2)
    # Salt
    coords = [np.random.randint(0, d, n_salt) for d in img.shape]
    img[coords[0], coords[1]] = 1.0
    # Pepper
    coords = [np.random.randint(0, d, n_pepper) for d in img.shape]
    img[coords[0], coords[1]] = 0.0
    return img


def corrupt_image(img):
    """Apply noise corruption to a clean image.
    Always Gaussian noise; optionally rectangular masking and salt-and-pepper."""
    noisy = add_gaussian_noise(img)
    if random.random() < MASK_PROB:
        noisy = add_rect_mask(noisy)
    if random.random() < SP_PROB:
        noisy = add_salt_pepper(noisy)
    return np.clip(noisy, 0.0, 1.0).astype(np.float32)


def corrupt_tensor(img_tensor):
    """Apply noise to a (1,H,W) tensor. Returns (1,H,W) tensor."""
    img_np = img_tensor.squeeze(0).cpu().numpy()
    noisy_np = corrupt_image(img_np)
    return torch.from_numpy(noisy_np).unsqueeze(0)


# ── Geometric augmentation (applied before noise) ──

def random_crop(img, size):
    h, w = img.shape
    if h < size or w < size:
        pad_h = max(0, size - h)
        pad_w = max(0, size - w)
        img = np.pad(img, ((0, pad_h), (0, pad_w)), mode='reflect')
        h, w = img.shape
    y = random.randint(0, h - size)
    x = random.randint(0, w - size)
    return img[y:y+size, x:x+size]


def center_crop(img, size):
    h, w = img.shape
    if h < size or w < size:
        pad_h = max(0, size - h)
        pad_w = max(0, size - w)
        img = np.pad(img, ((pad_h//2, pad_h - pad_h//2),
                           (pad_w//2, pad_w - pad_w//2)), mode='reflect')
        h, w = img.shape
    y = (h - size) // 2
    x = (w - size) // 2
    return img[y:y+size, x:x+size]


# ── Datasets ──

class MRIDAEDataset(Dataset):
    """Training dataset: returns (noisy, clean, label).
    Geometric augmentation + noise corruption."""
    def __init__(self, paths, labels, label_map, img_size=192, name=""):
        self.paths = paths
        self.label_map = label_map
        self.labels = [label_map[l] for l in labels]
        self.img_size = img_size
        print(f"  [{name}] {len(paths)} slices | Labels: {label_map}")

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = np.load(self.paths[idx]).astype(np.float32)

        # Geometric augmentation
        img = random_crop(img, self.img_size)
        if random.random() < 0.5:
            img = np.flip(img, axis=1).copy()
        if random.random() < 0.5:
            img = np.flip(img, axis=0).copy()

        clean = np.clip(img, 0.0, 1.0).astype(np.float32)
        noisy = corrupt_image(clean)

        clean_t = torch.from_numpy(clean).unsqueeze(0)  # (1,H,W)
        noisy_t = torch.from_numpy(noisy).unsqueeze(0)  # (1,H,W)
        return noisy_t, clean_t, self.labels[idx]


class CleanTransform:
    """No augmentation — just convert to tensor."""
    def __call__(self, x):
        return torch.from_numpy(x).unsqueeze(0).float()


class MRICleanDataset(Dataset):
    """Clean images for feature extraction (kNN scoring).
    No crop — encoder uses AdaptiveAvgPool for variable sizes."""
    def __init__(self, paths, labels, label_map, transform=None, name=""):
        self.paths = paths
        self.label_map = label_map
        self.labels = [label_map[l] for l in labels]
        self.transform = transform or CleanTransform()
        print(f"  [{name}] {len(paths)} slices | Labels: {label_map}")

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = np.load(self.paths[idx]).astype(np.float32)
        return self.transform(img), self.labels[idx]


class MRIReconDataset(Dataset):
    """Center-cropped images for reconstruction scoring.
    Fixed 192×192 so per-pixel MSE is well-defined."""
    def __init__(self, paths, labels, label_map, img_size=192, name=""):
        self.paths = paths
        self.label_map = label_map
        self.labels = [label_map[l] for l in labels]
        self.img_size = img_size
        print(f"  [{name}] {len(paths)} slices | Labels: {label_map}")

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = np.load(self.paths[idx]).astype(np.float32)
        img = center_crop(img, self.img_size)
        img = np.clip(img, 0.0, 1.0).astype(np.float32)
        return torch.from_numpy(img).unsqueeze(0), self.labels[idx]

## Model Architecture

### Encoder: 4-block CNN → 512-d bottleneck
- Each block: Conv3×3→BN→LeakyReLU→Conv3×3→BN→LeakyReLU→MaxPool2×2
- Channels: 1→64→128→256→512
- Spatial: 192→96→48→24→12 → AdaptiveAvgPool(1) → 512-d
- **No skip connections** — forces information through bottleneck

### Decoder: 4-block CNN → 192×192 output
- Each block: Upsample(2)→Conv3×3→BN→ReLU→Conv3×3→BN→ReLU
- Channels: 512→256→128→64→1
- Final layer: Sigmoid (output in [0,1])

### Design Rationale
- No skip connections = model must learn compact bottleneck representation
- Bilinear upsampling (not transposed conv) = avoids checkerboard artifacts
- LeakyReLU in encoder = better gradient flow in deeper layers

In [ ]:
# ══════════════════════════════════════════════
# Model Definition — Denoising Autoencoder
# ══════════════════════════════════════════════

def enc_block(in_c, out_c):
    return nn.Sequential(
        nn.Conv2d(in_c, out_c, 3, 1, 1), nn.BatchNorm2d(out_c), nn.LeakyReLU(0.2, True),
        nn.Conv2d(out_c, out_c, 3, 1, 1), nn.BatchNorm2d(out_c), nn.LeakyReLU(0.2, True),
        nn.MaxPool2d(2),
    )


def dec_block(in_c, out_c):
    return nn.Sequential(
        nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
        nn.Conv2d(in_c, out_c, 3, 1, 1), nn.BatchNorm2d(out_c), nn.ReLU(True),
        nn.Conv2d(out_c, out_c, 3, 1, 1), nn.BatchNorm2d(out_c), nn.ReLU(True),
    )


class DAEEncoder(nn.Module):
    """4-block convolutional encoder → 512-d bottleneck."""
    def __init__(self):
        super().__init__()
        self.block1 = enc_block(1, 64)
        self.block2 = enc_block(64, 128)
        self.block3 = enc_block(128, 256)
        self.block4 = enc_block(256, 512)
        self.pool = nn.AdaptiveAvgPool2d(1)

    def forward(self, x):
        x = self.block1(x)   # (B, 64, H/2, W/2)
        x = self.block2(x)   # (B, 128, H/4, W/4)
        x = self.block3(x)   # (B, 256, H/8, W/8)
        x = self.block4(x)   # (B, 512, H/16, W/16)
        return x

    def get_features(self, x):
        """Extract 512-d bottleneck features for kNN scoring."""
        return self.pool(self.forward(x)).flatten(1)  # (B, 512)


class DAEDecoder(nn.Module):
    """4-block convolutional decoder → 192×192 output."""
    def __init__(self):
        super().__init__()
        self.block1 = dec_block(512, 256)
        self.block2 = dec_block(256, 128)
        self.block3 = dec_block(128, 64)
        self.block4 = nn.Sequential(
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
            nn.Conv2d(64, 32, 3, 1, 1), nn.BatchNorm2d(32), nn.ReLU(True),
            nn.Conv2d(32, 1, 3, 1, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        x = self.block1(x)   # (B, 256, H/8, W/8)
        x = self.block2(x)   # (B, 128, H/4, W/4)
        x = self.block3(x)   # (B, 64, H/2, W/2)
        x = self.block4(x)   # (B, 1, H, W)
        return x


class DenoisingAutoencoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = DAEEncoder()
        self.decoder = DAEDecoder()

    def forward(self, noisy):
        """noisy → encoder → decoder → reconstruction.
        Returns: (recon, per_image_mse)"""
        latent = self.encoder(noisy)
        recon = self.decoder(latent)
        return recon


def count_params(model):
    return sum(p.numel() for p in model.parameters())


# Architecture check
_model = DenoisingAutoencoder()
print(f"Encoder params:  {count_params(_model.encoder):,} ({count_params(_model.encoder)/1e6:.1f}M)")
print(f"Decoder params:  {count_params(_model.decoder):,} ({count_params(_model.decoder)/1e6:.1f}M)")
print(f"Total params:    {count_params(_model):,} ({count_params(_model)/1e6:.1f}M)")

with torch.no_grad():
    _inp = torch.randn(2, 1, 192, 192)
    _out = _model(_inp)
    print(f"Input:  {_inp.shape}")
    print(f"Output: {_out.shape}")
    _feat = _model.encoder.get_features(_inp)
    print(f"Features: {_feat.shape}")
del _model, _inp, _out, _feat

In [ ]:
# ══════════════════════════════════════════════
# Training Infrastructure
# ══════════════════════════════════════════════
TOTAL_TIME_BUDGET = 12 * 3600
SESSION_START = time.time()


def log_time_budget(phase=""):
    elapsed = time.time() - SESSION_START
    remaining = TOTAL_TIME_BUDGET - elapsed
    print(f"⏱ [{phase}] Elapsed: {elapsed/3600:.2f}h | Remaining: {remaining/3600:.2f}h "
          f"| Budget used: {100*elapsed/TOTAL_TIME_BUDGET:.1f}%")
    if remaining < 1800:
        print("  ⚠️ WARNING: Less than 30 min remaining!")
    return remaining


def log_gpu_memory(prefix=""):
    if device == "cuda":
        alloc = torch.cuda.memory_allocated() / 1e9
        peak = torch.cuda.max_memory_allocated() / 1e9
        total = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"  🖥 GPU [{prefix}]: {alloc:.2f}/{total:.1f} GB (peak: {peak:.2f} GB)")


def clear_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
    log_gpu_memory("after clear")
    log_time_budget("GPU cleared")


def train_epoch(model, loader, optimizer, scaler):
    model.train()
    total_loss = 0.0
    num_batches = len(loader)
    log_interval = max(1, num_batches // 5)
    optimizer.zero_grad()
    epoch_start = time.time()

    for step, (noisy, clean, _) in enumerate(loader):
        noisy, clean = noisy.to(device), clean.to(device)

        with torch.amp.autocast("cuda", enabled=(device == "cuda")):
            recon = model(noisy)
            loss = F.mse_loss(recon, clean) / GRAD_ACCUM

        scaler.scale(loss).backward()

        if (step + 1) % GRAD_ACCUM == 0 or (step + 1) == num_batches:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        total_loss += loss.item() * GRAD_ACCUM

        if (step + 1) % log_interval == 0:
            avg_loss = total_loss / (step + 1)
            elapsed = time.time() - epoch_start
            batch_rate = (step + 1) / elapsed
            eta = (num_batches - step - 1) / batch_rate
            print(f"    batch {step+1}/{num_batches} | loss={avg_loss:.6f} | "
                  f"{batch_rate:.1f} batch/s | ETA: {eta:.0f}s", end="\r")

    print(" " * 80, end="\r")
    return total_loss / num_batches


@torch.no_grad()
def eval_epoch(model, loader):
    model.eval()
    total_loss = 0.0
    for noisy, clean, _ in loader:
        noisy, clean = noisy.to(device), clean.to(device)
        with torch.amp.autocast("cuda", enabled=(device == "cuda")):
            recon = model(noisy)
            loss = F.mse_loss(recon, clean)
        total_loss += loss.item()
    return total_loss / len(loader)


@torch.no_grad()
def extract_features(encoder, loader):
    """Extract L2-normalized encoder bottleneck features."""
    encoder.eval()
    feats, labels = [], []
    for x, y in tqdm(loader, desc="Extracting features"):
        x = x.to(device)
        with torch.amp.autocast("cuda", enabled=(device == "cuda")):
            f = encoder.get_features(x).float()
        f = F.normalize(f, dim=1)
        feats.append(f.cpu().numpy())
        labels.append(y.numpy())
    return np.concatenate(feats), np.concatenate(labels)


@torch.no_grad()
def compute_recon_scores(model, loader, num_trials=RECON_TRIALS):
    """Reconstruction anomaly score: average per-image MSE over multiple noise trials.
    Parallels MAE multi-mask scoring — same RECON_TRIALS=10."""
    model.eval()
    all_scores = []
    for clean_imgs, _ in tqdm(loader, desc="Recon scoring"):
        clean_imgs = clean_imgs.to(device)
        batch_scores = torch.zeros(clean_imgs.shape[0], device=device)

        for _ in range(num_trials):
            # Create noisy versions on CPU, then move to GPU
            noisy_batch = torch.stack([corrupt_tensor(img) for img in clean_imgs.cpu()]).to(device)
            with torch.amp.autocast("cuda", enabled=(device == "cuda")):
                recon = model(noisy_batch)
            # Per-image MSE
            per_img_mse = ((recon.float() - clean_imgs.float()) ** 2).mean(dim=[1, 2, 3])
            batch_scores += per_img_mse

        batch_scores /= num_trials
        all_scores.append(batch_scores.cpu().numpy())

    return np.concatenate(all_scores)

In [ ]:
# ══════════════════════════════════════════════
# Full Training Loop
# ══════════════════════════════════════════════
def train_dae(train_loader, val_loader, save_dir, model_name="DAE"):
    os.makedirs(save_dir, exist_ok=True)

    remaining = log_time_budget(f"START {model_name}")
    if remaining < 3600:
        print("  ⚠️ CRITICAL: Less than 1h remaining!")

    print(f"\n{'='*60}")
    print(f"  🚀 Training: {model_name}")
    print(f"  📁 Save dir: {save_dir}")
    print(f"  📊 Train batches: {len(train_loader)}")
    print(f"  ⚙️  Config: {EPOCHS}ep, BS={BATCH_SIZE}, accum={GRAD_ACCUM}")
    print(f"{'='*60}\n")

    model = DenoisingAutoencoder().to(device)
    total_params = count_params(model)
    print(f"  Parameters: {total_params:,} ({total_params/1e6:.1f}M)")
    log_gpu_memory("model loaded")

    optimizer = torch.optim.AdamW(
        model.parameters(), lr=BASE_LR, weight_decay=WEIGHT_DECAY,
    )

    def lr_lambda(epoch):
        if epoch < WARMUP_EPOCHS:
            return (epoch + 1) / WARMUP_EPOCHS
        progress = (epoch - WARMUP_EPOCHS) / max(EPOCHS - WARMUP_EPOCHS, 1)
        return 0.5 * (1.0 + math.cos(math.pi * progress))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    scaler = torch.amp.GradScaler("cuda", enabled=(device == "cuda"))

    history = {"train_loss": [], "val_loss": [], "lr": [], "epoch_time": []}
    best_val_loss = float("inf")
    total_start = time.time()

    for epoch in range(EPOCHS):
        epoch_start = time.time()

        t_loss = train_epoch(model, train_loader, optimizer, scaler)
        v_loss = eval_epoch(model, val_loader)
        scheduler.step()

        epoch_time = time.time() - epoch_start
        lr_now = scheduler.get_last_lr()[0]
        history["train_loss"].append(t_loss)
        history["val_loss"].append(v_loss)
        history["lr"].append(lr_now)
        history["epoch_time"].append(epoch_time)

        elapsed_session = time.time() - SESSION_START
        remaining_session = TOTAL_TIME_BUDGET - elapsed_session

        print(f"  [{model_name}] Epoch {epoch+1:2d}/{EPOCHS} │ "
              f"train={t_loss:.6f} val={v_loss:.6f} │ lr={lr_now:.6f} │ "
              f"{epoch_time:.0f}s/ep │ budget left: {remaining_session/3600:.2f}h")

        if v_loss < best_val_loss:
            best_val_loss = v_loss
            torch.save({
                "epoch": epoch + 1,
                "model": model.state_dict(),
                "val_loss": v_loss,
            }, os.path.join(save_dir, "best.pt"))
            print(f"       ↑ New best! (val_loss={v_loss:.6f})")

        if (epoch + 1) % 5 == 0:
            torch.save({
                "epoch": epoch + 1,
                "model": model.state_dict(),
            }, os.path.join(save_dir, f"epoch_{epoch+1}.pt"))
            print(f"       💾 Checkpoint: epoch_{epoch+1}.pt")
            log_gpu_memory(f"epoch {epoch+1}")

        if remaining_session < 1800 and epoch < EPOCHS - 1:
            print(f"  ⚠️ TIME: {remaining_session/60:.0f} min left! Stopping at epoch {epoch+1}.")
            break

    total_time = time.time() - total_start
    torch.save({
        "epoch": epoch + 1,
        "model": model.state_dict(),
        "history": history,
    }, os.path.join(save_dir, "final.pt"))

    print(f"\n  ✅ {model_name} complete in {total_time/60:.1f} min ({total_time/3600:.2f}h)")
    print(f"     Best val loss: {best_val_loss:.6f}")
    print(f"     Avg epoch time: {np.mean(history['epoch_time']):.1f}s")
    log_time_budget(f"END {model_name}")

    best_ckpt = torch.load(os.path.join(save_dir, "best.pt"),
                           map_location=device, weights_only=False)
    model.load_state_dict(best_ckpt["model"])

    return model, history

In [ ]:
# ══════════════════════════════════════════════
# Full Analysis Pipeline
# ══════════════════════════════════════════════
def run_full_analysis(model, train_clean_loader, val_clean_loader,
                      train_recon_loader, val_recon_loader,
                      val_clean_dataset, history, save_dir, model_name):
    analysis_start = time.time()
    log_time_budget(f"ANALYSIS START {model_name}")

    print(f"\n{'='*60}")
    print(f"  📈 Analysis: {model_name}")
    print(f"{'='*60}")

    # ── 1. Training Curves ──
    print("\n[1/12] Training curves...")
    fig, axes = plt.subplots(1, 4, figsize=(22, 5))
    fig.suptitle(f"{model_name} — Training Curves", fontsize=14, y=1.02)

    axes[0].plot(history["train_loss"], label="Train", linewidth=2)
    axes[0].plot(history["val_loss"], label="Val", linewidth=2)
    axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("MSE Loss")
    axes[0].set_title("Loss"); axes[0].legend(); axes[0].grid(True, alpha=0.3)

    axes[1].plot(history["lr"], linewidth=2, color="green")
    axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("LR")
    axes[1].set_title("Learning Rate"); axes[1].grid(True, alpha=0.3)

    gap = [v - t for t, v in zip(history["train_loss"], history["val_loss"])]
    axes[2].plot(gap, linewidth=2, color="red")
    axes[2].axhline(0, color="gray", ls="--", alpha=0.5)
    axes[2].set_xlabel("Epoch"); axes[2].set_ylabel("Val - Train")
    axes[2].set_title("Generalization Gap"); axes[2].grid(True, alpha=0.3)

    axes[3].plot(history["epoch_time"], linewidth=2, color="brown")
    axes[3].set_xlabel("Epoch"); axes[3].set_ylabel("Seconds")
    axes[3].set_title("Epoch Time"); axes[3].grid(True, alpha=0.3)

    fig.tight_layout()
    fig.savefig(os.path.join(save_dir, "training_curves.png"), dpi=150, bbox_inches="tight")
    plt.show()

    # ── 2. Reconstruction Visualization ──
    print("[2/12] Reconstruction visualization...")
    model.eval()
    sample_batch = next(iter(val_recon_loader))
    sample_clean = sample_batch[0][:8].to(device)
    sample_noisy = torch.stack([corrupt_tensor(img) for img in sample_clean.cpu()]).to(device)
    with torch.no_grad(), torch.amp.autocast("cuda", enabled=(device == "cuda")):
        sample_recon = model(sample_noisy)

    fig, axes = plt.subplots(3, 8, figsize=(20, 7.5))
    fig.suptitle(f"{model_name} — Denoising Examples", fontsize=14, y=1.02)
    for i in range(8):
        axes[0][i].imshow(sample_clean[i, 0].cpu(), cmap='gray', vmin=0, vmax=1)
        axes[0][i].set_title('Clean', fontsize=8); axes[0][i].axis('off')
        axes[1][i].imshow(sample_noisy[i, 0].cpu(), cmap='gray', vmin=0, vmax=1)
        axes[1][i].set_title('Noisy', fontsize=8); axes[1][i].axis('off')
        axes[2][i].imshow(sample_recon[i, 0].cpu().float(), cmap='gray', vmin=0, vmax=1)
        mse_val = F.mse_loss(sample_recon[i].float(), sample_clean[i].float()).item()
        axes[2][i].set_title(f'Recon (MSE={mse_val:.4f})', fontsize=7); axes[2][i].axis('off')
    fig.tight_layout()
    fig.savefig(os.path.join(save_dir, "reconstruction_viz.png"), dpi=150, bbox_inches="tight")
    plt.show()

    # ── 3. Error Map Visualization ──
    print("[3/12] Error maps...")
    fig, axes = plt.subplots(2, 8, figsize=(20, 5))
    fig.suptitle(f"{model_name} — Pixel-wise Error Maps", fontsize=14, y=1.02)
    for i in range(8):
        err = ((sample_recon[i, 0].cpu().float() - sample_clean[i, 0].cpu().float()) ** 2).numpy()
        axes[0][i].imshow(sample_clean[i, 0].cpu(), cmap='gray', vmin=0, vmax=1)
        axes[0][i].set_title('Clean', fontsize=8); axes[0][i].axis('off')
        axes[1][i].imshow(err, cmap='hot', vmin=0, vmax=err.max())
        axes[1][i].set_title(f'Error (max={err.max():.3f})', fontsize=7); axes[1][i].axis('off')
    fig.tight_layout()
    fig.savefig(os.path.join(save_dir, "error_maps.png"), dpi=150, bbox_inches="tight")
    plt.show()

    # ── 4. Feature Extraction ──
    print("[4/12] Extracting features...")
    feat_start = time.time()
    train_feats, train_labels = extract_features(model.encoder, train_clean_loader)
    val_feats, val_labels = extract_features(model.encoder, val_clean_loader)
    print(f"  Train: {train_feats.shape}, Val: {val_feats.shape} ({time.time()-feat_start:.1f}s)")

    # ── 5. Collapse Check ──
    print("[5/12] Collapse check...")
    n_check = min(2000, len(train_feats))
    for name, feats in [("Train", train_feats[:n_check]), ("Val", val_feats[:n_check])]:
        std = np.std(feats, axis=0).mean()
        sim_mat = np.dot(feats, feats.T)
        off_diag = sim_mat[np.triu_indices_from(sim_mat, k=1)]
        mean_sim = off_diag.mean()
        s_std = "✅" if std > 0.01 else "⚠️ COLLAPSE"
        s_sim = "✅" if mean_sim < 0.9 else "⚠️ HIGH"
        print(f"  {name}: STD={std:.4f} {s_std} | Mean cos sim={mean_sim:.4f} {s_sim}")

    # ── 6. Reconstruction Anomaly Scores ──
    print("[6/12] Reconstruction anomaly scores (multi-trial)...")
    recon_start = time.time()
    train_recon = compute_recon_scores(model, train_recon_loader, RECON_TRIALS)
    val_recon = compute_recon_scores(model, val_recon_loader, RECON_TRIALS)
    print(f"  Train recon — mean: {train_recon.mean():.6f}, std: {train_recon.std():.6f}")
    print(f"  Val recon   — mean: {val_recon.mean():.6f}, std: {val_recon.std():.6f}")
    print(f"  ({time.time()-recon_start:.1f}s)")

    # ── 7. kNN Anomaly Scores ──
    print("[7/12] kNN anomaly scores...")
    nn_model = NearestNeighbors(n_neighbors=K_NEIGHBORS, metric="cosine")
    nn_model.fit(train_feats)
    train_knn = nn_model.kneighbors(train_feats)[0].mean(axis=1)
    val_knn = nn_model.kneighbors(val_feats)[0].mean(axis=1)
    print(f"  Train kNN — mean: {train_knn.mean():.4f}, std: {train_knn.std():.4f}")
    print(f"  Val kNN   — mean: {val_knn.mean():.4f}, std: {val_knn.std():.4f}")

    # ── 8. Score Distributions ──
    print("[8/12] Score distributions...")
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle(f"{model_name} — Anomaly Score Distributions", fontsize=14, y=1.02)
    axes[0][0].hist(train_recon, bins=60, alpha=0.7, color="steelblue", edgecolor="white")
    axes[0][0].set_title("Train — Recon MSE"); axes[0][0].set_xlabel("MSE")
    axes[0][1].hist(val_recon, bins=60, alpha=0.7, color="coral", edgecolor="white")
    axes[0][1].set_title("Val — Recon MSE"); axes[0][1].set_xlabel("MSE")
    axes[1][0].hist(train_knn, bins=60, alpha=0.7, color="steelblue", edgecolor="white")
    axes[1][0].set_title("Train — kNN Distance"); axes[1][0].set_xlabel("Cosine Dist")
    axes[1][1].hist(val_knn, bins=60, alpha=0.7, color="coral", edgecolor="white")
    axes[1][1].set_title("Val — kNN Distance"); axes[1][1].set_xlabel("Cosine Dist")
    fig.tight_layout()
    fig.savefig(os.path.join(save_dir, "anomaly_scores.png"), dpi=150, bbox_inches="tight")
    plt.show()

    # ── 9. Threshold Analysis ──
    print("[9/12] Threshold analysis...")
    print("  Reconstruction-based:")
    for pct in [90, 95, 97.5, 99]:
        thresh = np.percentile(train_recon, pct)
        flagged = (val_recon > thresh).sum()
        print(f"    p{pct}: thresh={thresh:.6f} → {flagged}/{len(val_recon)} val flagged "
              f"({100*flagged/len(val_recon):.1f}%)")
    print("  kNN-based:")
    for pct in [90, 95, 97.5, 99]:
        thresh = np.percentile(train_knn, pct)
        flagged = (val_knn > thresh).sum()
        print(f"    p{pct}: thresh={thresh:.4f} → {flagged}/{len(val_knn)} val flagged "
              f"({100*flagged/len(val_knn):.1f}%)")

    # ── 10. t-SNE ──
    print("[10/12] t-SNE...")
    tsne_start = time.time()
    N_TSNE = min(3000, len(val_feats))
    tsne = TSNE(n_components=2, perplexity=30, random_state=SEED, max_iter=1000)
    emb = tsne.fit_transform(val_feats[:N_TSNE])
    labels_sub = val_labels[:N_TSNE]
    knn_sub = val_knn[:N_TSNE]
    recon_sub = val_recon[:N_TSNE]
    print(f"  t-SNE done ({time.time()-tsne_start:.1f}s)")

    fig, axes = plt.subplots(1, 4, figsize=(22, 5.5))
    fig.suptitle(f"{model_name} — t-SNE Embeddings", fontsize=14, y=1.02)
    axes[0].scatter(emb[:, 0], emb[:, 1], s=3, alpha=0.5, c="steelblue")
    axes[0].set_title("Unsupervised"); axes[0].set_xticks([]); axes[0].set_yticks([])
    sc1 = axes[1].scatter(emb[:, 0], emb[:, 1], s=3, alpha=0.5, c=labels_sub, cmap="coolwarm")
    axes[1].set_title("By Label"); axes[1].set_xticks([]); axes[1].set_yticks([])
    plt.colorbar(sc1, ax=axes[1], shrink=0.8)
    sc2 = axes[2].scatter(emb[:, 0], emb[:, 1], s=3, alpha=0.5, c=knn_sub, cmap="YlOrRd")
    axes[2].set_title("kNN Score"); axes[2].set_xticks([]); axes[2].set_yticks([])
    plt.colorbar(sc2, ax=axes[2], shrink=0.8)
    sc3 = axes[3].scatter(emb[:, 0], emb[:, 1], s=3, alpha=0.5, c=recon_sub, cmap="YlOrRd")
    axes[3].set_title("Recon MSE Score"); axes[3].set_xticks([]); axes[3].set_yticks([])
    plt.colorbar(sc3, ax=axes[3], shrink=0.8)
    fig.tight_layout()
    fig.savefig(os.path.join(save_dir, "tsne.png"), dpi=150, bbox_inches="tight")
    plt.show()

    # ── 11. Cosine Similarity Distribution ──
    print("[11/12] Cosine similarity distribution...")
    subset = torch.tensor(val_feats[:2000])
    sim_matrix = torch.mm(subset, subset.T).numpy()
    upper_tri = sim_matrix[np.triu_indices_from(sim_matrix, k=1)]
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.hist(upper_tri, bins=100, color="mediumpurple", edgecolor="white", alpha=0.8)
    ax.axvline(upper_tri.mean(), color="red", ls="--", label=f"Mean: {upper_tri.mean():.3f}")
    ax.set_title(f"{model_name} — Pairwise Cosine Similarity")
    ax.set_xlabel("Cosine Similarity"); ax.legend()
    fig.tight_layout()
    fig.savefig(os.path.join(save_dir, "similarity_dist.png"), dpi=150, bbox_inches="tight")
    plt.show()

    # ── 12. Top Anomalies ──
    print("[12/12] Top anomalies...")
    fig, axes = plt.subplots(2, 5, figsize=(15, 6))
    fig.suptitle(f"{model_name} — Top-10 Anomalies (by recon MSE)", fontsize=14)
    top_idx = np.argsort(val_recon)[-10:][::-1]
    for i, idx in enumerate(top_idx):
        ax = axes[i // 5][i % 5]
        img = np.load(val_clean_dataset.paths[idx])
        ax.imshow(img, cmap="gray", vmin=0, vmax=1)
        ax.set_title(f"MSE={val_recon[idx]:.4f}\nkNN={val_knn[idx]:.4f}", fontsize=7)
        ax.axis("off")
    fig.tight_layout()
    fig.savefig(os.path.join(save_dir, "top_anomalies.png"), dpi=150, bbox_inches="tight")
    plt.show()

    # ── Save ──
    np.savez(
        os.path.join(save_dir, "scoring.npz"),
        train_recon=train_recon, val_recon=val_recon,
        train_knn=train_knn, val_knn=val_knn,
        train_feats=train_feats, val_feats=val_feats,
        train_labels=train_labels, val_labels=val_labels,
        threshold_recon_p95=np.percentile(train_recon, 95),
        threshold_knn_p95=np.percentile(train_knn, 95),
    )

    print(f"\n  ✅ Analysis complete in {(time.time()-analysis_start)/60:.1f} min")
    print(f"  📁 Files: {os.listdir(save_dir)}")
    log_time_budget(f"ANALYSIS END {model_name}")

    return train_feats, val_feats, train_recon, val_recon, train_knn, val_knn

In [ ]:
# ══════════════════════════════════════════════
# Discover dataset paths
# ══════════════════════════════════════════════
KNEE_ROOT = "/kaggle/input/datasets/kaustubhratna/fast-mri-preprocessed-kaust"
BRAIN_ROOT = "/kaggle/input/datasets/kaustubhratna/preprocessed-ixi-brain"

def show_tree(path, prefix="", max_depth=3, depth=0):
    if depth >= max_depth:
        return
    try:
        entries = sorted(os.listdir(path))
    except (PermissionError, FileNotFoundError) as e:
        print(f"{prefix}⚠️ {e}")
        return
    dirs = [e for e in entries if os.path.isdir(os.path.join(path, e))]
    files = [e for e in entries if os.path.isfile(os.path.join(path, e))]
    if files:
        npy = [f for f in files if f.endswith('.npy')]
        if npy:
            print(f"{prefix}📄 {len(npy)} .npy files (e.g., {npy[0]})")
        for f in [x for x in files if not x.endswith('.npy')][:3]:
            print(f"{prefix}📄 {f}")
    for d in dirs:
        print(f"{prefix}📁 {d}/")
        show_tree(os.path.join(path, d), prefix + "    ", max_depth, depth + 1)

print("=" * 50)
print("KNEE Dataset:")
print("=" * 50)
show_tree(KNEE_ROOT)

print(f"\n{'='*50}")
print("BRAIN Dataset:")
print("=" * 50)
show_tree(BRAIN_ROOT)

In [ ]:
# ══════════════════════════════════════════════
# Configure Paths + Balanced Brain Sampling
# ══════════════════════════════════════════════
KNEE_TRAIN = os.path.join(KNEE_ROOT, "train")
KNEE_VAL   = os.path.join(KNEE_ROOT, "val")
BRAIN_TRAIN = os.path.join(BRAIN_ROOT, "train")
BRAIN_VAL   = os.path.join(BRAIN_ROOT, "val")

print("Knee (all):")
knee_train_paths, knee_train_labels = get_balanced_paths(KNEE_TRAIN)
knee_val_paths, knee_val_labels = get_balanced_paths(KNEE_VAL)
print(f"  Train: {len(knee_train_paths)} | Val: {len(knee_val_paths)}")

print(f"\nBrain (balanced to match knee = {len(knee_train_paths)}):")
print("  Train:")
brain_train_paths, brain_train_labels = get_balanced_paths(
    BRAIN_TRAIN, target_total=len(knee_train_paths))
print("  Val:")
brain_val_paths, brain_val_labels = get_balanced_paths(
    BRAIN_VAL, target_total=len(knee_val_paths))
print(f"  Train: {len(brain_train_paths)} | Val: {len(brain_val_paths)}")

knee_label_map = {l: i for i, l in enumerate(sorted(set(knee_train_labels)))}
brain_label_map = {l: i for i, l in enumerate(sorted(set(brain_train_labels)))}
all_labels = sorted(set(knee_train_labels + brain_train_labels))
combined_label_map = {l: i for i, l in enumerate(all_labels)}

print(f"\nLabel maps:")
print(f"  Knee: {knee_label_map}")
print(f"  Brain: {brain_label_map}")
print(f"  Combined: {combined_label_map}")

---
# Part 1: Knee Only (FastMRI)

In [ ]:
# ══════════════════════════════════════════════
# PART 1: KNEE — Data Loaders
# ══════════════════════════════════════════════
print("Loading Knee datasets...")

knee_train_ds = MRIDAEDataset(knee_train_paths, knee_train_labels, knee_label_map,
                               img_size=IMG_SIZE, name="Knee-Train")
knee_val_ds = MRIDAEDataset(knee_val_paths, knee_val_labels, knee_label_map,
                             img_size=IMG_SIZE, name="Knee-Val")

knee_train_loader = DataLoader(knee_train_ds, batch_size=BATCH_SIZE, shuffle=True,
                               num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
knee_val_loader = DataLoader(knee_val_ds, batch_size=BATCH_SIZE, shuffle=False,
                             num_workers=NUM_WORKERS, pin_memory=True)

# Clean loaders for feature extraction
knee_train_clean = MRICleanDataset(knee_train_paths, knee_train_labels, knee_label_map,
                                    name="Knee-Train-Clean")
knee_val_clean = MRICleanDataset(knee_val_paths, knee_val_labels, knee_label_map,
                                  name="Knee-Val-Clean")
knee_train_clean_loader = DataLoader(knee_train_clean, batch_size=BATCH_SIZE * 2, shuffle=False,
                                     num_workers=NUM_WORKERS, pin_memory=True)
knee_val_clean_loader = DataLoader(knee_val_clean, batch_size=BATCH_SIZE * 2, shuffle=False,
                                   num_workers=NUM_WORKERS, pin_memory=True)

# Recon loaders for reconstruction scoring (center-cropped)
knee_train_recon = MRIReconDataset(knee_train_paths, knee_train_labels, knee_label_map,
                                    img_size=IMG_SIZE, name="Knee-Train-Recon")
knee_val_recon = MRIReconDataset(knee_val_paths, knee_val_labels, knee_label_map,
                                  img_size=IMG_SIZE, name="Knee-Val-Recon")
knee_train_recon_loader = DataLoader(knee_train_recon, batch_size=BATCH_SIZE, shuffle=False,
                                     num_workers=NUM_WORKERS, pin_memory=True)
knee_val_recon_loader = DataLoader(knee_val_recon, batch_size=BATCH_SIZE, shuffle=False,
                                   num_workers=NUM_WORKERS, pin_memory=True)

print(f"\nTrain batches: {len(knee_train_loader)}")
print(f"Knee train: {len(knee_train_ds)} | Val: {len(knee_val_ds)}")

In [ ]:
# ══════════════════════════════════════════════
# PART 1: KNEE — Noise Corruption Preview
# ══════════════════════════════════════════════
fig, axes = plt.subplots(3, 6, figsize=(15, 7.5))
fig.suptitle("Knee — DAE Noise Corruption Preview", fontsize=14, y=1.02)

for row in range(3):
    idx = random.randint(0, len(knee_train_recon) - 1)
    clean_img, _ = knee_train_recon[idx]
    raw = clean_img.squeeze(0).numpy()

    axes[row][0].imshow(raw, cmap="gray", vmin=0, vmax=1)
    axes[row][0].set_title("Clean", fontweight="bold", fontsize=9)
    axes[row][0].axis("off")

    for col in range(1, 6):
        noisy = corrupt_image(raw)
        axes[row][col].imshow(noisy, cmap="gray", vmin=0, vmax=1)
        axes[row][col].set_title(f"Noisy {col}", fontsize=9)
        axes[row][col].axis("off")

fig.tight_layout()
plt.show()

In [ ]:
# ══════════════════════════════════════════════
# PART 1: KNEE — Train DAE
# ══════════════════════════════════════════════
KNEE_SAVE = "/kaggle/working/checkpoints/dae_knee"
knee_model, knee_history = train_dae(
    knee_train_loader, knee_val_loader, KNEE_SAVE, model_name="DAE-Knee")

In [ ]:
# ══════════════════════════════════════════════
# PART 1: KNEE — Full Analysis
# ══════════════════════════════════════════════
knee_results = run_full_analysis(
    knee_model, knee_train_clean_loader, knee_val_clean_loader,
    knee_train_recon_loader, knee_val_recon_loader,
    knee_val_clean, knee_history, KNEE_SAVE, "DAE-Knee")

In [ ]:
del knee_model
del knee_train_loader, knee_val_loader
del knee_train_clean_loader, knee_val_clean_loader
del knee_train_recon_loader, knee_val_recon_loader
clear_gpu()

---
# Part 2: Brain Only (IXI — Balanced T1 + T2)

In [ ]:
# ══════════════════════════════════════════════
# PART 2: BRAIN — Data Loaders (balanced)
# ══════════════════════════════════════════════
print("Loading Brain datasets (balanced)...")

brain_train_ds = MRIDAEDataset(brain_train_paths, brain_train_labels, brain_label_map,
                                img_size=IMG_SIZE, name="Brain-Train")
brain_val_ds = MRIDAEDataset(brain_val_paths, brain_val_labels, brain_label_map,
                              img_size=IMG_SIZE, name="Brain-Val")

brain_train_loader = DataLoader(brain_train_ds, batch_size=BATCH_SIZE, shuffle=True,
                                num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
brain_val_loader = DataLoader(brain_val_ds, batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=True)

brain_train_clean = MRICleanDataset(brain_train_paths, brain_train_labels, brain_label_map,
                                     name="Brain-Train-Clean")
brain_val_clean = MRICleanDataset(brain_val_paths, brain_val_labels, brain_label_map,
                                   name="Brain-Val-Clean")
brain_train_clean_loader = DataLoader(brain_train_clean, batch_size=BATCH_SIZE * 2, shuffle=False,
                                      num_workers=NUM_WORKERS, pin_memory=True)
brain_val_clean_loader = DataLoader(brain_val_clean, batch_size=BATCH_SIZE * 2, shuffle=False,
                                    num_workers=NUM_WORKERS, pin_memory=True)

brain_train_recon = MRIReconDataset(brain_train_paths, brain_train_labels, brain_label_map,
                                     img_size=IMG_SIZE, name="Brain-Train-Recon")
brain_val_recon = MRIReconDataset(brain_val_paths, brain_val_labels, brain_label_map,
                                   img_size=IMG_SIZE, name="Brain-Val-Recon")
brain_train_recon_loader = DataLoader(brain_train_recon, batch_size=BATCH_SIZE, shuffle=False,
                                      num_workers=NUM_WORKERS, pin_memory=True)
brain_val_recon_loader = DataLoader(brain_val_recon, batch_size=BATCH_SIZE, shuffle=False,
                                    num_workers=NUM_WORKERS, pin_memory=True)

print(f"\nTrain batches: {len(brain_train_loader)}")
print(f"Brain train: {len(brain_train_ds)} | Val: {len(brain_val_ds)}")

In [ ]:
# ══════════════════════════════════════════════
# PART 2: BRAIN — Train DAE
# ══════════════════════════════════════════════
BRAIN_SAVE = "/kaggle/working/checkpoints/dae_brain"
brain_model, brain_history = train_dae(
    brain_train_loader, brain_val_loader, BRAIN_SAVE, model_name="DAE-Brain")

In [ ]:
# ══════════════════════════════════════════════
# PART 2: BRAIN — Full Analysis
# ══════════════════════════════════════════════
brain_results = run_full_analysis(
    brain_model, brain_train_clean_loader, brain_val_clean_loader,
    brain_train_recon_loader, brain_val_recon_loader,
    brain_val_clean, brain_history, BRAIN_SAVE, "DAE-Brain")

In [ ]:
del brain_model
del brain_train_loader, brain_val_loader
del brain_train_clean_loader, brain_val_clean_loader
del brain_train_recon_loader, brain_val_recon_loader
clear_gpu()

---
# Part 3: Combined (Knee + Balanced Brain)

In [ ]:
# ══════════════════════════════════════════════
# PART 3: COMBINED — Data Loaders
# ══════════════════════════════════════════════
print("Loading Combined (Knee + balanced Brain) datasets...")

comb_train_paths = knee_train_paths + brain_train_paths
comb_train_labels = knee_train_labels + brain_train_labels
comb_val_paths = knee_val_paths + brain_val_paths
comb_val_labels = knee_val_labels + brain_val_labels

comb_train_ds = MRIDAEDataset(comb_train_paths, comb_train_labels, combined_label_map,
                               img_size=IMG_SIZE, name="Combined-Train")
comb_val_ds = MRIDAEDataset(comb_val_paths, comb_val_labels, combined_label_map,
                             img_size=IMG_SIZE, name="Combined-Val")

comb_train_loader = DataLoader(comb_train_ds, batch_size=BATCH_SIZE, shuffle=True,
                               num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
comb_val_loader = DataLoader(comb_val_ds, batch_size=BATCH_SIZE, shuffle=False,
                             num_workers=NUM_WORKERS, pin_memory=True)

comb_train_clean = MRICleanDataset(comb_train_paths, comb_train_labels, combined_label_map,
                                    name="Combined-Train-Clean")
comb_val_clean = MRICleanDataset(comb_val_paths, comb_val_labels, combined_label_map,
                                  name="Combined-Val-Clean")
comb_train_clean_loader = DataLoader(comb_train_clean, batch_size=BATCH_SIZE * 2, shuffle=False,
                                     num_workers=NUM_WORKERS, pin_memory=True)
comb_val_clean_loader = DataLoader(comb_val_clean, batch_size=BATCH_SIZE * 2, shuffle=False,
                                   num_workers=NUM_WORKERS, pin_memory=True)

comb_train_recon = MRIReconDataset(comb_train_paths, comb_train_labels, combined_label_map,
                                    img_size=IMG_SIZE, name="Combined-Train-Recon")
comb_val_recon = MRIReconDataset(comb_val_paths, comb_val_labels, combined_label_map,
                                  img_size=IMG_SIZE, name="Combined-Val-Recon")
comb_train_recon_loader = DataLoader(comb_train_recon, batch_size=BATCH_SIZE, shuffle=False,
                                     num_workers=NUM_WORKERS, pin_memory=True)
comb_val_recon_loader = DataLoader(comb_val_recon, batch_size=BATCH_SIZE, shuffle=False,
                                   num_workers=NUM_WORKERS, pin_memory=True)

print(f"\nCombined train: {len(comb_train_ds)} slices, {len(comb_train_loader)} batches")
print(f"Combined val: {len(comb_val_ds)} slices")

In [ ]:
# ══════════════════════════════════════════════
# PART 3: COMBINED — Train DAE
# ══════════════════════════════════════════════
COMBINED_SAVE = "/kaggle/working/checkpoints/dae_combined"
comb_model, comb_history = train_dae(
    comb_train_loader, comb_val_loader, COMBINED_SAVE, model_name="DAE-Combined")

In [ ]:
# ══════════════════════════════════════════════
# PART 3: COMBINED — Full Analysis
# ══════════════════════════════════════════════
comb_results = run_full_analysis(
    comb_model, comb_train_clean_loader, comb_val_clean_loader,
    comb_train_recon_loader, comb_val_recon_loader,
    comb_val_clean, comb_history, COMBINED_SAVE, "DAE-Combined")

In [ ]:
del comb_model
del comb_train_loader, comb_val_loader
del comb_train_clean_loader, comb_val_clean_loader
del comb_train_recon_loader, comb_val_recon_loader
clear_gpu()

---
# Summary & Comparison

In [ ]:
# ══════════════════════════════════════════════
# Cross-Model Comparison
# ══════════════════════════════════════════════
print("\n" + "=" * 80)
print("  DAE MODEL COMPARISON SUMMARY")
print("=" * 80)

results = {}
for name, save_dir in [("Knee", KNEE_SAVE), ("Brain", BRAIN_SAVE), ("Combined", COMBINED_SAVE)]:
    data = np.load(os.path.join(save_dir, "scoring.npz"), allow_pickle=True)
    results[name] = {
        "train_recon": data["train_recon"],
        "val_recon": data["val_recon"],
        "train_knn": data["train_knn"],
        "val_knn": data["val_knn"],
        "val_feats": data["val_feats"],
        "thresh_recon": float(data["threshold_recon_p95"]),
        "thresh_knn": float(data["threshold_knn_p95"]),
    }

print(f"\n  Reconstruction-based scoring:")
print(f"  {'Model':<12} {'Train μ':<12} {'Val μ':<12} {'Val σ':<12} {'p95 Thresh':<12} {'Val>p95':}")
print("  " + "-" * 72)
for name, r in results.items():
    tr, vr = r["train_recon"], r["val_recon"]
    flagged = (vr > r["thresh_recon"]).sum()
    print(f"  {name:<12} {tr.mean():<12.6f} {vr.mean():<12.6f} {vr.std():<12.6f} "
          f"{r['thresh_recon']:<12.6f} {flagged}/{len(vr)} ({100*flagged/len(vr):.1f}%)")

print(f"\n  kNN-based scoring:")
print(f"  {'Model':<12} {'Train μ':<10} {'Val μ':<10} {'Val σ':<10} {'p95 Thresh':<12} {'Val>p95':}")
print("  " + "-" * 68)
for name, r in results.items():
    tr, vr = r["train_knn"], r["val_knn"]
    flagged = (vr > r["thresh_knn"]).sum()
    print(f"  {name:<12} {tr.mean():<10.4f} {vr.mean():<10.4f} {vr.std():<10.4f} "
          f"{r['thresh_knn']:<12.4f} {flagged}/{len(vr)} ({100*flagged/len(vr):.1f}%)")

print(f"\n  Collapse check:")
print(f"  {'Model':<12} {'Feature STD':<14} {'Mean Cos Sim':<14} {'Status'}")
print("  " + "-" * 54)
for name, r in results.items():
    feats = r["val_feats"][:2000]
    std = np.std(feats, axis=0).mean()
    sim = np.dot(feats, feats.T)
    off = sim[np.triu_indices_from(sim, k=1)].mean()
    status = "✅ Good" if off < 0.9 else "⚠️ High sim"
    print(f"  {name:<12} {std:<14.4f} {off:<14.4f} {status}")

In [ ]:
# ══════════════════════════════════════════════
# Comparative Plots
# ══════════════════════════════════════════════
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle("DAE — Model Comparison", fontsize=14, y=1.02)

colors = {"Knee": "tab:blue", "Brain": "tab:orange", "Combined": "tab:green"}

# Load histories
histories = {}
for name, save_dir in [("Knee", KNEE_SAVE), ("Brain", BRAIN_SAVE), ("Combined", COMBINED_SAVE)]:
    ckpt = torch.load(os.path.join(save_dir, "final.pt"), map_location="cpu", weights_only=False)
    histories[name] = ckpt["history"]

# Row 1: Loss, Recon scores, kNN scores
for name, h in histories.items():
    axes[0][0].plot(h['train_loss'], label=f"{name} train", linewidth=2, color=colors[name])
    axes[0][0].plot(h['val_loss'], label=f"{name} val", linewidth=2, color=colors[name], linestyle='--')
axes[0][0].set_title('MSE Loss'); axes[0][0].set_xlabel('Epoch')
axes[0][0].legend(fontsize=7); axes[0][0].grid(True, alpha=0.3)

for name, r in results.items():
    axes[0][1].hist(r['val_recon'], bins=50, alpha=0.5, label=name, color=colors[name])
axes[0][1].set_title('Val Recon MSE Scores'); axes[0][1].set_xlabel('MSE')
axes[0][1].legend(); axes[0][1].grid(True, alpha=0.3)

for name, r in results.items():
    axes[0][2].hist(r['val_knn'], bins=50, alpha=0.5, label=name, color=colors[name])
axes[0][2].set_title('Val kNN Scores'); axes[0][2].set_xlabel('Cosine Dist')
axes[0][2].legend(); axes[0][2].grid(True, alpha=0.3)

# Row 2: Epoch time, recon vs kNN correlation, LR schedule
for name, h in histories.items():
    axes[1][0].bar(name, np.mean(h['epoch_time']), color=colors[name])
axes[1][0].set_title('Avg Epoch Time (s)'); axes[1][0].grid(True, alpha=0.3)

for name, r in results.items():
    axes[1][1].scatter(r['val_knn'][:1000], r['val_recon'][:1000],
                       s=3, alpha=0.3, label=name, color=colors[name])
axes[1][1].set_title('kNN vs Recon Score'); axes[1][1].set_xlabel('kNN'); axes[1][1].set_ylabel('Recon MSE')
axes[1][1].legend(); axes[1][1].grid(True, alpha=0.3)

for name, h in histories.items():
    axes[1][2].plot(h['lr'], linewidth=2, color=colors[name], label=name)
axes[1][2].set_title('LR Schedule'); axes[1][2].set_xlabel('Epoch')
axes[1][2].legend(); axes[1][2].grid(True, alpha=0.3)

fig.tight_layout()
fig.savefig('dae_comparison_plots.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# # ══════════════════════════════════════════════
# # Zip & Download All Checkpoints
# # ══════════════════════════════════════════════
# import shutil
# from IPython.display import FileLink, display

# for name, save_dir in [("knee", KNEE_SAVE), ("brain", BRAIN_SAVE), ("combined", COMBINED_SAVE)]:
#     zip_path = shutil.make_archive(f"dae_{name}_checkpoints", "zip",
#                                    root_dir=".", base_dir=save_dir)
#     size_mb = os.path.getsize(zip_path) / 1e6
#     print(f"  {name}: {zip_path} ({size_mb:.1f} MB)")
#     display(FileLink(zip_path))

# all_zip = shutil.make_archive("dae_all_checkpoints", "zip",
#                               root_dir=".", base_dir="/kaggle/working/checkpoints")
# print(f"\n  ALL: {all_zip} ({os.path.getsize(all_zip)/1e6:.1f} MB)")
# display(FileLink(all_zip))

# log_time_budget("SESSION COMPLETE")